# Audit S3 Minute Coverage

Проверяет дневные parquet-партиции в S3 и показывает:

- полностью отсутствующие дни по каждому источнику;
- дни, где внутри parquet не хватает минут;
- лишние/дублирующиеся минуты, если они есть.

По умолчанию проверяется `ADAUSDT`, `1m`, все группы из `features/` и `raw/sinthetic_data`. Горизонты `return_15m_forward`, `return_20m_forward`, `return_25m_forward` можно оставить в аудите или исключить через настройки ниже.

In [1]:
import io
import os
import re
from dataclasses import dataclass
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

## Settings

In [2]:
BUCKET = "binance-data-downloader"
SYMBOL = "ADAUSDT"
INTERVAL = "1m"

START_DATE = "2020-02-01"
END_DATE = "2026-02-01"

# Если нужно проверить вообще все return horizons, поставь False.
EXCLUDE_UNUSED_RETURN_HORIZONS = True
EXCLUDED_FEATURE_GROUPS = {"return_15m_forward", "return_20m_forward", "return_25m_forward"}

# Полезно для быстрой проверки. Для полного аудита оставь None.
MAX_DAYS = None

# Сколько конкретных пропущенных минут показывать в таблицах.
MAX_MISSING_MINUTES_TO_SHOW = 20

In [3]:
load_dotenv(dotenv_path=".env")

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

## Helpers

In [4]:
@dataclass(frozen=True)
class Source:
    name: str
    kind: str
    prefix: str


def list_feature_groups():
    groups = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET, Prefix="features/", Delimiter="/"):
        for item in page.get("CommonPrefixes", []):
            group = item["Prefix"].strip("/").split("/")[-1]
            if EXCLUDE_UNUSED_RETURN_HORIZONS and group in EXCLUDED_FEATURE_GROUPS:
                continue
            groups.append(group)
    return sorted(groups)


def build_sources():
    sources = []
    for group in list_feature_groups():
        prefix = f"features/{group}/symbol={SYMBOL}/interval={INTERVAL}/"
        sources.append(Source(name=group, kind="features", prefix=prefix))
    sources.append(Source(name="sinthetic_data", kind="raw", prefix="raw/sinthetic_data/"))
    return sources


def key_for_date(source, date):
    if source.name == "sinthetic_data":
        return f"raw/sinthetic_data/date={date}/data.parquet"
    return f"{source.prefix}date={date}/data.parquet"


def list_existing_dates(source):
    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET, Prefix=source.prefix):
        for obj in page.get("Contents", []):
            match = re.search(r"date=([0-9]{4}-[0-9]{2}-[0-9]{2})", obj["Key"])
            if match:
                dates.add(match.group(1))
    return dates


def read_parquet(key):
    try:
        body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    except ClientError as exc:
        if exc.response.get("Error", {}).get("Code") in {"404", "NoSuchKey"}:
            return None
        raise
    return pd.read_parquet(io.BytesIO(body))


def extract_minute_timestamps(df):
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], utc=True)
    elif "open_time" in df.columns:
        ts = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    else:
        raise ValueError("No timestamp/open_time column found")
    return ts.dt.floor("min")


def expected_minutes_for_date(date):
    start = pd.Timestamp(date, tz="UTC")
    return pd.date_range(start, periods=1440, freq="min")


def compact_minutes(minutes, limit=MAX_MISSING_MINUTES_TO_SHOW):
    values = [str(x) for x in minutes[:limit]]
    if len(minutes) > limit:
        values.append(f"... +{len(minutes) - limit} more")
    return values

## Source Inventory

In [5]:
sources = build_sources()
pd.DataFrame([source.__dict__ for source in sources])

,name,kind,prefix
0,aggression_features,features,features/aggression_features/symbol=ADAUSDT/interval=1m/
1,btc_features,features,features/btc_features/symbol=ADAUSDT/interval=1m/
2,intraminute_dynamics,features,features/intraminute_dynamics/symbol=ADAUSDT/interval=1m/
3,intraminute_features,features,features/intraminute_features/symbol=ADAUSDT/interval=1m/
4,intraminute_segments,features,features/intraminute_segments/symbol=ADAUSDT/interval=1m/
5,price_pressure,features,features/price_pressure/symbol=ADAUSDT/interval=1m/
6,return_1m_forward,features,features/return_1m_forward/symbol=ADAUSDT/interval=1m/
7,trade_distribution,features,features/trade_distribution/symbol=ADAUSDT/interval=1m/
8,trades_minute_level,features,features/trades_minute_level/symbol=ADAUSDT/interval=1m/
9,sinthetic_data,raw,raw/sinthetic_data/


## Fully Missing Days

День считается полностью пропущенным для источника, если нет parquet-файла `date=YYYY-MM-DD/data.parquet`.

In [6]:
expected_dates = pd.date_range(START_DATE, END_DATE, freq="D").strftime("%Y-%m-%d").tolist()
if MAX_DAYS is not None:
    expected_dates = expected_dates[:MAX_DAYS]

existing_by_source = {source.name: list_existing_dates(source) for source in sources}

missing_day_rows = []
for source in sources:
    existing = existing_by_source[source.name]
    missing = [date for date in expected_dates if date not in existing]
    missing_day_rows.append(
        {
            "source": source.name,
            "kind": source.kind,
            "expected_days": len(expected_dates),
            "existing_days": len([date for date in expected_dates if date in existing]),
            "missing_days_count": len(missing),
            "missing_days": missing,
        }
    )

missing_days_df = pd.DataFrame(missing_day_rows).sort_values(["missing_days_count", "source"], ascending=[False, True])
missing_days_df

,source,kind,expected_days,existing_days,missing_days_count,missing_days
2,intraminute_dynamics,features,2193,2190,3,"[2020-03-15, 2022-04-19, 2022-05-01]"
0,aggression_features,features,2193,2193,0,[]
1,btc_features,features,2193,2193,0,[]
3,intraminute_features,features,2193,2193,0,[]
4,intraminute_segments,features,2193,2193,0,[]
5,price_pressure,features,2193,2193,0,[]
6,return_1m_forward,features,2193,2193,0,[]
9,sinthetic_data,raw,2193,2193,0,[]
7,trade_distribution,features,2193,2193,0,[]
8,trades_minute_level,features,2193,2193,0,[]


## Days With Missing Minutes

Для каждого существующего дневного файла сравнивается набор минут с полным UTC-днём `00:00..23:59`.

In [7]:
coverage_rows = []

for source in sources:
    print(f"audit {source.name}")
    for date in expected_dates:
        if date not in existing_by_source[source.name]:
            continue

        key = key_for_date(source, date)
        df = read_parquet(key)
        if df is None:
            continue

        timestamps = extract_minute_timestamps(df)
        expected = expected_minutes_for_date(date)
        unique_timestamps = pd.DatetimeIndex(timestamps.dropna().unique()).sort_values()

        missing_minutes = expected.difference(unique_timestamps)
        extra_minutes = unique_timestamps.difference(expected)
        duplicate_count = int(timestamps.duplicated().sum())

        if len(missing_minutes) or len(extra_minutes) or duplicate_count:
            coverage_rows.append(
                {
                    "source": source.name,
                    "kind": source.kind,
                    "date": date,
                    "rows": len(df),
                    "unique_minutes": len(unique_timestamps),
                    "missing_minutes_count": len(missing_minutes),
                    "extra_minutes_count": len(extra_minutes),
                    "duplicate_minutes_count": duplicate_count,
                    "missing_minutes_sample": compact_minutes(missing_minutes),
                    "extra_minutes_sample": compact_minutes(extra_minutes),
                    "key": key,
                }
            )

coverage_issues_df = pd.DataFrame(coverage_rows)
coverage_issues_df

audit aggression_features
audit btc_features
audit intraminute_dynamics
audit intraminute_features
audit intraminute_segments
audit price_pressure
audit return_1m_forward
audit trade_distribution
audit trades_minute_level
audit sinthetic_data


,source,kind,date,rows,unique_minutes,missing_minutes_count,extra_minutes_count,duplicate_minutes_count,missing_minutes_sample,extra_minutes_sample,key
0,aggression_features,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/aggression_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
1,btc_features,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/btc_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
2,intraminute_dynamics,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/intraminute_dynamics/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
3,intraminute_features,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
4,intraminute_segments,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/intraminute_segments/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
5,price_pressure,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/price_pressure/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
6,return_1m_forward,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/return_1m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
7,return_1m_forward,features,2026-02-01,1439,1439,1,0,0,[2026-02-01 23:59:00+00:00],[],features/return_1m_forward/symbol=ADAUSDT/interval=1m/date=2026-02-01/data.parquet
8,trade_distribution,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
9,trades_minute_level,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/trades_minute_level/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet


## Summary

In [8]:
if coverage_issues_df.empty:
    print("No minute-level coverage issues found in existing files.")
else:
    summary = (
        coverage_issues_df
        .groupby(["source", "kind"], as_index=False)
        .agg(
            days_with_issues=("date", "count"),
            total_missing_minutes=("missing_minutes_count", "sum"),
            total_extra_minutes=("extra_minutes_count", "sum"),
            total_duplicate_minutes=("duplicate_minutes_count", "sum"),
        )
        .sort_values(["days_with_issues", "total_missing_minutes"], ascending=[False, False])
    )
    display(summary)

,source,kind,days_with_issues,total_missing_minutes,total_extra_minutes,total_duplicate_minutes
6,return_1m_forward,features,2,2,0,0
0,aggression_features,features,1,1,0,0
1,btc_features,features,1,1,0,0
2,intraminute_dynamics,features,1,1,0,0
3,intraminute_features,features,1,1,0,0
4,intraminute_segments,features,1,1,0,0
5,price_pressure,features,1,1,0,0
7,trade_distribution,features,1,1,0,0
8,trades_minute_level,features,1,1,0,0


## Most Important Tables

In [9]:
display(missing_days_df[missing_days_df["missing_days_count"] > 0])

if not coverage_issues_df.empty:
    display(
        coverage_issues_df
        .sort_values(["missing_minutes_count", "duplicate_minutes_count", "source", "date"], ascending=[False, False, True, True])
        .reset_index(drop=True)
    )

,source,kind,expected_days,existing_days,missing_days_count,missing_days
2,intraminute_dynamics,features,2193,2190,3,"[2020-03-15, 2022-04-19, 2022-05-01]"


,source,kind,date,rows,unique_minutes,missing_minutes_count,extra_minutes_count,duplicate_minutes_count,missing_minutes_sample,extra_minutes_sample,key
0,aggression_features,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/aggression_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
1,btc_features,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/btc_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
2,intraminute_dynamics,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/intraminute_dynamics/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
3,intraminute_features,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
4,intraminute_segments,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/intraminute_segments/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
5,price_pressure,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/price_pressure/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
6,return_1m_forward,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/return_1m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
7,return_1m_forward,features,2026-02-01,1439,1439,1,0,0,[2026-02-01 23:59:00+00:00],[],features/return_1m_forward/symbol=ADAUSDT/interval=1m/date=2026-02-01/data.parquet
8,trade_distribution,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
9,trades_minute_level,features,2020-02-01,1439,1439,1,0,0,[2020-02-01 00:00:00+00:00],[],features/trades_minute_level/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet


## Save Reports

In [10]:
report_dir = Path("coverage_reports")
report_dir.mkdir(exist_ok=True)

missing_days_df.to_csv(report_dir / "s3_missing_full_days.csv", index=False)
coverage_issues_df.to_csv(report_dir / "s3_missing_minutes.csv", index=False)

print(f"saved: {report_dir / 's3_missing_full_days.csv'}")
print(f"saved: {report_dir / 's3_missing_minutes.csv'}")

saved: coverage_reports\s3_missing_full_days.csv
saved: coverage_reports\s3_missing_minutes.csv
